# Урок 01 - Вступ до агентів ШІ

Ласкаво просимо на перший урок курсу **Агенти ШІ для початківців**!

**Агент ШІ** — це програма, яка використовує велику мовну модель (LLM) як рушій логіки й може виконувати *дії* у реальному світі — викликати API, запитувати бази даних або запускати код — щоб досягти мети від імені користувача.

У цьому зошиті ви створите свого першого агента: **Туристичного агента**, що рекомендує місця для відпочинку. По ходу уроку ви дізнаєтеся, як:

1. Підключитись до Microsoft Foundry Agent Service за допомогою **Microsoft Agent Framework**.
2. Додати агенту **інструмент** — просту функцію Python, яку він може викликати.
3. Запустити агента та проаналізувати його відповідь.
4. Потоково отримувати відповідь агента по одному токену.


## Налаштування

Перед запуском цього ноутбука переконайтеся, що:

1. **У вас є проект Microsoft Foundry** з розгорнутою моделлю чат-бота (наприклад, `gpt-5-mini`).
2. **Ви увійшли через Azure CLI** — виконайте `az login` у вашому терміналі.
3. **Встановлені необхідні змінні середовища:**
   - `AZURE_AI_PROJECT_ENDPOINT` — кінцева точка вашого проекту Microsoft Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — назва вашої розгорнутої моделі.

Наведена нижче комірка встановлює необхідні вам пакети Python.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Створення вашого першого агента

Агенту потрібно дві речі:

- **Інструкції**, які кажуть йому *хто він* і *як поводитися* (системний запит).
- **Інструменти** — функції Python, прикрашені `@tool`, які агент може викликати, щоб отримати інформацію або виконати дії.

Нижче ми визначаємо простий інструмент, який повертає список популярних місць для відпочинку. Агент використовуватиме цей інструмент, коли користувач проситиме рекомендації з подорожей.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Потокові відповіді

Для більш інтерактивного досвіду ви можете **потоково** отримувати відповідь агента. Замість того, щоб чекати повної відповіді, агент надає текстові фрагменти у міру їх генерації. Це особливо корисно в чат-інтерфейсах, де потрібно відображати результат у реальному часі.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Підсумок

У цьому уроці ви дізналися, як:

- **Створити провайдера**, який підключається до Microsoft Foundry Agent Service за допомогою `FoundryChatClient`.
- **Визначити інструмент** за допомогою декоратора `@tool`, щоб агент міг викликати ваші функції Python.
- **Запустити агента** з повідомленням користувача і вивести його відповідь.
- **Виводити відповіді в потоці** для реального часу.

У наступному уроці ми глибше дослідимо агентські фреймворки і навчимося надавати агентам потужніші інструменти та можливості багатокрокового логічного мислення.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу штучного інтелекту для перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується професійний людський переклад. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
